In [1]:
def riboDensityAtEachPosition(inputFile,outputFile):
    density_dict = {}
    # open file
    with open(inputFile,'r') as input:
        for line in input:
            if line.startswith('@'):
                continue 
            fileds = line.split()
            if fileds[2] == '*':
                continue
            id = fileds[2]
            align_pos = int(fileds[3])
            length = len(fileds[9])
            if 22 < length < 36:
                # sam is 1-based format
                end5 = align_pos
                end3 = end5 + length - 1
                # shift +- 11nt
                centerEnd5 = end5 + 11
                centerEnd3 = end3 - 11
                centerLength = centerEnd3 - centerEnd5 + 1
                # ribo density
                for elem in range(centerEnd5, centerEnd3 + 1):
                    key = ':'.join([id,str(elem)])
                    if key in density_dict:
                        density_dict[key] += (1.0 / centerLength)
                    else:
                        density_dict[key] = (1.0 / centerLength)
            else:
                pass

    # total densitys
    total_density = sum(density_dict.values())

    # RPM normalization
    density_RPM_dict = {}
    for key,val in density_dict.items():
        val1 = (val/total_density)*1000000
        density_RPM_dict[key] = [val,val1]

    # sort dict
    pList = list(density_RPM_dict.items())
    pList.sort()
    # output
    outFileP = open(outputFile, 'w')
    for J in pList:
        outFileP.write('\t'.join([J[0].split(':')[0],J[0].split(':')[1],str(J[1][0]),str(J[1][1])]) + '\n')
    outFileP.close()

In [2]:
import os

# make folder
os.mkdir('./3.ribo-density-data')

In [4]:
sample = []
for i in [range(27,45),range(49,61)]:
    for j in i:
        samp = ''.join(['../5.map-data/','SRR74712',str(j),'.sam'])
        sample.append(samp)

# output name
outputName = ['FAS1-trans-rep1','FAS1-inter-rep1','FAS1-trans-rep2','FAS1-inter-rep2',
                'FAS2-trans-rep1','FAS2-inter-rep1','FAS2-trans-rep2','FAS2-inter-rep2',
                'FAS1-MPTdel-trans-rep1','FAS1-MPTdel-inter-rep1','FAS1-MPTdel-trans-rep2','FAS1-MPTdel-trans-rep3','FAS1-MPTdel-inter-rep2','FAS1-MPTdel-inter-rep3',
                'FAS2-MPTdel-trans-rep1','FAS2-MPTdel-inter-rep1','FAS2-MPTdel-trans-rep2','FAS2-MPTdel-inter-rep2',
                'GUS1-trans-rep1','GUS1-inter-rep1','GUS1-trans-rep2','GUS1-inter-rep2',
                'MES1-trans-rep1','MES1-inter-rep1','MES1-trans-rep2','MES1-inter-rep2',
                'ARC1-trans-rep1','ARC1-inter-rep1','ARC1-trans-rep2','ARC1-inter-rep2']

# run
for i in range(0,len(sample)):
    riboDensityAtEachPosition(sample[i],''.join(['3.ribo-density-data/',outputName[i],'.density.txt']))